# 編集距離と n-gram 言語モデルによる誤り検出・訂正

書籍 第3章「系列ラベリング問題の解き方」（3-3 誤り検出：誤り箇所の検出と訂正、p.104〜110）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、GECToR などの深層学習手法の前段階となる古典的・基礎的な仕組みを実装して確認します。

1. **編集距離（Levenshtein 距離）と隣接文字入れ替え（制約付き Damerau–Levenshtein 距離）**（3-3-1、脚注 42〜43）
2. **編集距離を用いたスペル訂正の候補ランキング**
3. **単語バイグラム言語モデルによる誤り検出**（3-3-2）
4. **編集距離と言語モデルを組み合わせた文脈スペル・誤字訂正（雑音のある通信路モデル）**（3-3）

## 1. 編集距離（Levenshtein 距離）の実装（3-3-1）

2つの文字列 $A$（長さ $m$）と $B$（長さ $n$）の間で、「挿入・削除・置換」の最小操作回数を動的計画法（計算量 $O(mn)$）で求めます。
また、書籍 3-3-1（脚注 42〜43）で紹介されているように、キーボード入力では `the` を `hte` のように隣り合う 2 文字を入れ替える（swap）ミスがよく起きるため、入れ替えもコスト 1 と数える **制約付き Damerau–Levenshtein 距離（OSA）** も実装して比較します。

In [1]:
import math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

def levenshtein_dp(a, b):
    '''Levenshtein 距離と DP テーブルを返す（計算量 O(mn)）'''
    m, n = len(a), len(b)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i, 0] = i
    for j in range(n + 1):
        dp[0, j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,       # 削除
                dp[i, j - 1] + 1,       # 挿入
                dp[i - 1, j - 1] + cost # 置換（一致なら 0）
            )
    return int(dp[m, n]), dp

def damerau_levenshtein(a, b):
    '''隣接文字の入れ替え（swap）もコスト 1 とする編集距離'''
    m, n = len(a), len(b)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i, 0] = i
    for j in range(n + 1):
        dp[0, j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                dp[i, j] = min(dp[i, j], dp[i - 2, j - 2] + 1)  # swap
    return int(dp[m, n])

dist, dp_table = levenshtein_dp("hte", "the")
print(f"Levenshtein('hte', 'the')         = {dist}")
print(f"OSA('hte', 'the') = {damerau_levenshtein('hte', 'the')}")

print("\nDP テーブル ('hte' -> 'the'):")
display(pd.DataFrame(dp_table, index=["#"] + list("hte"), columns=["#"] + list("the")))

Levenshtein('hte', 'the')         = 2
OSA('hte', 'the') = 1

DP テーブル ('hte' -> 'the'):


,#,t,h,e
#,0,1,2,3
h,1,1,1,2
t,2,1,2,2
e,3,2,2,2


---

## 2. 編集距離を使ったスペル訂正の候補ランキング

辞書（単語リスト）の中から、入力語との編集距離が小さい順に訂正候補を並べます。

In [2]:
vocab = ["the", "they", "there", "their", "them", "then", "heat", "hat", "language", "natural", "processing", "model"]

typos = ["hte", "langage", "natrual", "proccessing"]
for w in typos:
    ranked = sorted(vocab, key=lambda v: (damerau_levenshtein(w, v), levenshtein_dp(w, v)[0], v))[:3]
    scores = [(cand, damerau_levenshtein(w, cand), levenshtein_dp(w, cand)[0]) for cand in ranked]
    print(f"入力: {w:12s} -> 候補 (単語, OSA, Levenshtein): {scores}")

入力: hte          -> 候補 (単語, OSA, Levenshtein): [('the', 1, 2), ('hat', 2, 2), ('them', 2, 3)]
入力: langage      -> 候補 (単語, OSA, Levenshtein): [('language', 1, 1), ('hat', 6, 6), ('heat', 6, 6)]
入力: natrual      -> 候補 (単語, OSA, Levenshtein): [('natural', 1, 2), ('hat', 5, 5), ('language', 5, 5)]
入力: proccessing  -> 候補 (単語, OSA, Levenshtein): [('processing', 1, 1), ('model', 9, 9), ('their', 9, 9)]


---

## 3. n-gram 言語モデルによる誤り検出（3-3-2）

単語単位のスペルが正しくても、文脈に合わない誤用（例：`I like an apple` vs `I like a apple`、`自然言語処理` vs `自然言語処置`）は辞書引きだけでは検出できません。
ここでは Add-k スムージング付きの単語バイグラム言語モデルを構築し、文の**パープレキシティ（Perplexity; PPL）**（低いほどトークンあたりの予測確率が高い）を比較してみます。

In [3]:
train_sents = [
    "<s> i like an apple </s>",
    "<s> i eat an apple every day </s>",
    "<s> i like a banana </s>",
    "<s> she likes a book </s>",
    "<s> he reads a book every day </s>",
    "<s> we study natural language processing </s>",
    "<s> natural language processing is fun </s>",
] * 20

unigram_counts = Counter()
bigram_counts = defaultdict(Counter)
vocab_lm = set()

for s in train_sents:
    toks = s.split()
    vocab_lm.update(toks)
    for i, tok in enumerate(toks):
        unigram_counts[tok] += 1
        if i > 0:
            bigram_counts[toks[i - 1]][tok] += 1

def bigram_perplexity(sent, k=0.1):
    toks = ["<s>"] + sent.lower().split() + ["</s>"]
    V = len(vocab_lm)
    log_prob = 0.0
    n = len(toks) - 1
    for i in range(1, len(toks)):
        prev_w, curr_w = toks[i - 1], toks[i]
        num = bigram_counts[prev_w][curr_w] + k
        den = unigram_counts[prev_w] + k * V
        log_prob += math.log(num / den)
    return math.exp(-log_prob / n)

test_sents = [
    ("自然な文 1", "i like an apple"),
    ("誤りを含む文 1（冠詞ミス）", "i like a apple"),
    ("自然な文 2", "natural language processing is fun"),
    ("誤りを含む文 2（語順ミス）", "language natural is processing fun"),
]

rows = []
for label, s in test_sents:
    rows.append({"ケース": label, "文": s, "Perplexity (低いほど自然)": round(bigram_perplexity(s), 2)})
pd.DataFrame(rows)

,ケース,文,Perplexity (低いほど自然)
0,自然な文 1,i like an apple,1.77
1,誤りを含む文 1（冠詞ミス）,i like a apple,6.33
2,自然な文 2,natural language processing is fun,1.65
3,誤りを含む文 2（語順ミス）,language natural is processing fun,172.83


---

## 4. 編集距離と n-gram 言語モデルを組み合わせた文脈スペル・誤字訂正（3-3）

第 2 節の「編集距離のみ」の訂正は文脈を見ないため、同じ最小編集距離の候補が複数あるとき（例: `wat` → `cat` / `eat` / `hat` / `mat`）に適切な候補を選べず、また語彙に存在する**実単語誤り（real-word error）**（例: `i like a apple` の `a`、`she reads form a book` の `form`）も検出・訂正できません。一方、第 3 節の「言語モデルのみ」では誤りの検出（パープレキシティ上昇）はできても訂正候補を作れません。

そこで両者を統合する古典的手法が**雑音のある通信路モデル（Noisy Channel Model）**です。入力文 $x = (x_1, \dots, x_T)$ に対し、元の正しい文 $w = (w_1, \dots, w_T)$ を事後確率最大（MAP）で推定します。

$$\hat{w} = \arg\max_{w} P(w \mid x) = \arg\max_{w} P(x \mid w)\, P(w)$$

対数確率で書くと、**通信路（誤り）モデル**と**言語モデル**の和の最大化になります。

$$\log P(x \mid w) + \log P(w) = \sum_{t=1}^{T} \log P(x_t \mid w_t) + \sum_{t=1}^{T+1} \log P(w_t \mid w_{t-1})$$

- **通信路モデル $P(x_t \mid w_t)$**: 各単語位置で OSA 距離 $\le 1$ の語彙内単語を候補とし、誤字なし（$d=0$）の確率を $1 - \varepsilon$、編集距離 1 の誤字（$d=1$）の重みを $\varepsilon$（ここでは $\varepsilon = 0.05$）とします。
- **言語モデル $P(w_t \mid w_{t-1})$**: Add-$k$ スムージング（$k = 0.1$）付き単語バイグラム言語モデルを用います。
- **探索**: Viterbi アルゴリズムで文全体の対数スコア和が最大の候補系列を求めます。

In [4]:
nc_train_sents = [
    "<s> i like an apple </s>",
    "<s> i eat an apple every day </s>",
    "<s> i like a banana </s>",
    "<s> she likes a book </s>",
    "<s> he reads a book every day </s>",
    "<s> she reads from a book </s>",
    "<s> we study natural language processing </s>",
    "<s> natural language processing is fun </s>",
    "<s> a black cat sleeps on the mat </s>",
    "<s> the boy bought a red hat </s>",
    "<s> please fill out this form </s>",
    "<s> he opens a small box </s>",
] * 20

nc_uni = Counter()
nc_bi = defaultdict(Counter)
nc_vocab = set()
for s in nc_train_sents:
    toks = s.split()
    nc_vocab.update(toks)
    for i, tok in enumerate(toks):
        nc_uni[tok] += 1
        if i > 0:
            nc_bi[toks[i - 1]][tok] += 1

word_vocab = sorted(nc_vocab - {"<s>", "</s>"})

def log_bigram(prev_w, curr_w, k=0.1):
    V = len(nc_vocab)
    return math.log((nc_bi[prev_w][curr_w] + k) / (nc_uni[prev_w] + k * V))

def correct_min_edit(sent):
    '''編集距離のみ（語彙内にあれば距離0としてそのまま、なければOSA最小・辞書順）'''
    out = []
    for w in sent.lower().split():
        if w in nc_vocab:
            out.append(w)
        else:
            out.append(min(word_vocab, key=lambda v: (damerau_levenshtein(w, v), v)))
    return " ".join(out)

def correct_noisy_channel(sent, k=0.1, error_prob=0.05):
    '''OSA距離 <= 1 の候補集合上で Viterbi 探索を行い Noisy Channel 事後確率最大列を返す'''
    toks = sent.lower().split()
    cands_list = []
    for w in toks:
        cands = [(v, damerau_levenshtein(w, v)) for v in word_vocab if damerau_levenshtein(w, v) <= 1]
        if not cands:
            cands = [(w, 0)]
        cands_list.append(cands)

    dp = {}
    back = []
    for v, d in cands_list[0]:
        ch = math.log(1.0 - error_prob) if d == 0 else math.log(error_prob)
        dp[v] = ch + log_bigram("<s>", v, k)

    for t in range(1, len(toks)):
        new_dp = {}
        bp = {}
        for v, d in cands_list[t]:
            ch = math.log(1.0 - error_prob) if d == 0 else math.log(error_prob)
            best_score = -1e18
            best_prev = None
            for pv, pscore in dp.items():
                sc = pscore + ch + log_bigram(pv, v, k)
                if sc > best_score:
                    best_score = sc
                    best_prev = pv
            new_dp[v] = best_score
            bp[v] = best_prev
        dp = new_dp
        back.append(bp)

    best_last = max(dp.keys(), key=lambda v: dp[v] + log_bigram(v, "</s>", k))
    res = [best_last]
    cur = best_last
    for bp in reversed(back):
        cur = bp[cur]
        res.append(cur)
    res.reverse()
    return " ".join(res)

cases = [
    ("非単語誤字（頻出・右文脈）", "she reads a bok every day", "she reads a book every day"),
    ("隣接文字入れ替え（OSA=1）", "we study natrual language processing", "we study natural language processing"),
    ("非単語誤字（同距離候補の文脈解消）", "i wat an apple every day", "i eat an apple every day"),
    ("実単語誤り（冠詞 a / an）", "i like a apple", "i like an apple"),
    ("実単語誤り（入れ替え form / from）", "she reads form a book", "she reads from a book"),
    ("正しい文（過剰訂正しない）", "please fill out this form", "please fill out this form"),
]

comp_rows = []
for label, inp, gold in cases:
    ed_out = correct_min_edit(inp)
    nc_out = correct_noisy_channel(inp)
    comp_rows.append({
        "ケース": label,
        "入力文": inp,
        "編集距離のみ（OSA最小）": ed_out,
        "Noisy Channel（OSA+LM）": nc_out,
        "正解": gold,
        "ED一致": ed_out == gold,
        "NC一致": nc_out == gold,
    })

display(pd.DataFrame(comp_rows))

def candidate_breakdown(left_w, obs_w, right_w, k=0.1, error_prob=0.05):
    rows = []
    for v in word_vocab:
        d = damerau_levenshtein(obs_w, v)
        if d <= 1:
            ch = math.log(1.0 - error_prob) if d == 0 else math.log(error_prob)
            lm = log_bigram(left_w, v, k) + log_bigram(v, right_w, k)
            rows.append({
                "観測語": obs_w,
                "候補語": v,
                "OSA距離": d,
                "通信路 log P(x|w)": round(ch, 3),
                "文脈LM log P(w|前後)": round(lm, 3),
                "合計スコア": round(ch + lm, 3),
            })
    return pd.DataFrame(rows).sort_values("合計スコア", ascending=False).reset_index(drop=True)

print("候補別スコア内訳 1: 'i [wat] an apple every day'")
display(candidate_breakdown("i", "wat", "an"))

print("候補別スコア内訳 2: 'i like [a] apple'")
display(candidate_breakdown("like", "a", "apple"))

,ケース,入力文,編集距離のみ（OSA最小）,Noisy Channel（OSA+LM）,正解,ED一致,NC一致
0,非単語誤字（頻出・右文脈）,she reads a bok every day,she reads a book every day,she reads a book every day,she reads a book every day,True,True
1,隣接文字入れ替え（OSA=1）,we study natrual language processing,we study natural language processing,we study natural language processing,we study natural language processing,True,True
2,非単語誤字（同距離候補の文脈解消）,i wat an apple every day,i cat an apple every day,i eat an apple every day,i eat an apple every day,False,True
3,実単語誤り（冠詞 a / an）,i like a apple,i like a apple,i like an apple,i like an apple,False,True
4,実単語誤り（入れ替え form / from）,she reads form a book,she reads form a book,she reads from a book,she reads from a book,False,True
5,正しい文（過剰訂正しない）,please fill out this form,please fill out this form,please fill out this form,please fill out this form,True,True


候補別スコア内訳 1: 'i [wat] an apple every day'


,観測語,候補語,OSA距離,通信路 log P(x|w),文脈LM log P(w|前後),合計スコア
0,wat,eat,1,-2.996,-1.347,-4.343
1,wat,cat,1,-2.996,-11.954,-14.949
2,wat,hat,1,-2.996,-11.954,-14.949
3,wat,mat,1,-2.996,-11.954,-14.949


候補別スコア内訳 2: 'i like [a] apple'


,観測語,候補語,OSA距離,通信路 log P(x|w),文脈LM log P(w|前後),合計スコア
0,a,an,1,-2.996,-0.885,-3.881
1,a,a,0,-0.051,-8.062,-8.113
2,a,i,1,-2.996,-12.556,-15.552


結果のとおり、通信路モデルと言語モデルを組み合わせることで次の 3 点が同時に実現できています。

1. **同距離候補の曖昧性解消**: `wat` には OSA 距離 1 の候補として `cat` / `eat` / `hat` / `mat` が並び、編集距離だけでは辞書順先頭の `cat` を誤選択しますが、バイグラム文脈 `i [?] an` により `eat` が大幅に高いスコアを得て正しく訂正されます。
2. **実単語誤りの訂正**: `i like a apple` の `a` や `she reads form a book` の `form` は辞書に存在する語（距離 0）のため編集距離のみでは素通りしてしまいますが、文脈確率の差が誤りペナルティ（$\log \frac{1-\varepsilon}{\varepsilon}$）を上回るため、`an` や `from` に訂正されます。
3. **過剰訂正の抑制**: 正しい文 `please fill out this form` では、文脈にも合致する距離 0 の `form` が選ばれるため、不要な書き換えは起きません。